In [ ]:
import copy
import os
import sys

# from FocalLoss import FocalLoss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.utils.data as Data
from torch import nn
from tqdm import tqdm

from data_xrd_elements import Sub_XRDDataset_3attention_angle_abc, Sub_XRDDataset_3attention4test_angle_abc
from loss import SelectiveLoss
from scrnet_plus import RCNet

In [ ]:
def collate_fn(batch):
    features, targets, conf, cs, lattice, pg, objects, abc, angles = zip(*batch)  # 解包batch
    features = torch.stack(features)  # 把features变成一个batch的张量
    conf = torch.stack(conf)
    cs = torch.stack(cs)
    lattice = torch.stack(lattice)
    pg = torch.stack(pg)
    targets = torch.stack(targets)  # 标签也堆叠成张量
    angles = torch.stack(angles)
    abc = torch.stack(abc)
    return features, targets, conf, cs, lattice, pg, objects, abc, angles

In [ ]:
bs = 256  # 64 # 32 #
hs = 2 ** 8  # 2**9
n_layers = 2  #1
bidirect = False  # True
D = 2 if bidirect else 1
hid0 = D * n_layers
dvsid = [0, 1, 2, 3]
news = "angles_abc v7"
# data_root = "/home/light/PycharmProjects/material/resampled_test/merged_data_label_select.npy"


In [ ]:
def get_weights(w1, w2):
    # 使用 softmax 进行归一化
    weights = F.softmax(torch.tensor([w1, w2]), dim=0)
    return weights

In [ ]:
import numpy as np


def train(model, train_dataloader, val_dataloader, model_name, num_list=None, start=0, length=0, epochs=10):
    tqdm.write(f"{model_name}" + " start: " + str(start) + " length: " + str(length))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-5)
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-4, weight_decay=1e-4)
    class_count = num_list
    num_max = max(class_count)
    weights = [num_max / i for i in class_count]
    weights = torch.FloatTensor(weights)
    early_stop = 30
    # criterion = nn.CrossEntropyLoss(weights)
    criterion = SelectiveLoss(start, length, weight=weights)
    criterion_conf = nn.CrossEntropyLoss()
    criterion_abc_angles = nn.MSELoss()
    model = nn.DataParallel(model, output_device=dvsid).to(device)  #model #
    lossOfepochInTrain = []
    accOfepochInTrain = []
    lossOfepochInVal = []
    accOfepochInVal = []
    best_acc = 0.0
    min_val_loss = 9999
    best_avg_acc = 0
    best_model_wts = copy.deepcopy(model.module.state_dict()) if isinstance(model, nn.DataParallel) else copy.deepcopy(
        model.state_dict())

    nanLossCount = 0
    with torch.no_grad():
        model.eval()
        val_loss = 0
        val_correct = 0
        val_num = 0
        val_conf_correct = 0
        val_conf_num = 0
        for data, target, conf_target, cs, lattice, pg, _, abc, angles in tqdm(val_dataloader, desc=f"{model_name}",
                                                                               file=sys.stdout):
            data, target, conf_target, cs, lattice, pg, abc, angles = data.to(device), target.to(
                device), conf_target.to(
                device), cs.to(device), lattice.to(device), pg.to(device), abc.to(device), angles.to(device)
            data = data.unsqueeze(1)

            # cs = torch.zeros_like(cs,dtype=torch.float32)
            # lattice = torch.zeros_like(lattice,dtype=torch.float32)
            # pg = torch.zeros_like(pg,dtype=torch.float32)
            output, conf, abc_angles = model(data, cs, lattice, pg)
            pre = torch.argmax(output, dim=1)
            pre_conf = torch.argmax(conf, dim=1)
            #计算abc_angles和abc,angles的abc
            abc_angles_labels = torch.cat([abc, angles], dim=1)
            loss_abc_angles = criterion_abc_angles(abc_angles, abc_angles_labels) / (1000)
            loss = criterion(output, target)
            loss_conf = criterion_conf(conf, conf_target)

            loss_ = loss + loss_conf  + loss_abc_angles
            val_loss += loss_.item() * data.size(0)
            valid_mask = (target >= start) & (target < start + length)
            val_correct += torch.sum(pre[valid_mask] == target.data[valid_mask])
            val_num += data[valid_mask].size(0)

            val_conf_correct += torch.sum(pre_conf == conf_target)
            val_conf_num += data.size(0)
    min_val_loss = val_loss / val_num
    best_acc = val_correct / val_num
    best_avg_acc = (best_acc + (val_conf_correct / val_conf_num)) / 2
    tqdm.write(
        f"min_loss:{min_val_loss:.4f} train_correct:{best_acc:.4f}")
    for epoch in range(epochs):
        train_loss = 0
        train_correct = 0
        train_num = 0
        train_conf_correct = 0
        val_loss = 0
        val_correct = 0
        val_num = 0
        val_conf_correct = 0
        train_conf_num = 0
        val_conf_num = 0
        # best_avg_acc = 0
        model.train()
        for data, target, conf_target, cs, lattice, pg, _, abc, angles in tqdm(train_dataloader,
                                                                               desc=f"{model_name} epoch:{epoch + 1}/{epochs}",
                                                                               unit="batch",
                                                                               file=sys.stdout):
            data, target, conf_target, cs, lattice, pg, abc, angles = data.to(device), target.to(
                device), conf_target.to(
                device), cs.to(device), lattice.to(device), pg.to(device), abc.to(device), angles.to(device)
            optimizer.zero_grad()
            data = data.unsqueeze(1)
            # cs = torch.zeros_like(cs,dtype=torch.float32)
            # lattice = torch.zeros_like(lattice,dtype=torch.float32)
            # pg = torch.zeros_like(pg,dtype=torch.float32)
            output, conf, abc_angles = model(data, cs, lattice, pg)
            pre = torch.argmax(output, dim=1)
            conf_soma = torch.argmax(conf, dim=1)
            abc_angles_labels = torch.cat([abc, angles], dim=1)
            loss_abc_angles = criterion_abc_angles(abc_angles, abc_angles_labels) / (1000)
            loss = criterion(output, target)
            loss_conf = criterion_conf(conf, conf_target)
            loss_ = loss + loss_conf+ loss_abc_angles
            loss_.backward()
            optimizer.step()
            train_conf_correct += torch.sum(conf_soma == conf_target)
            train_loss += loss_.item() * data.size(0)
            valid_mask = (target >= start) & (target < start + length)
            train_correct += torch.sum(pre[valid_mask] == target.data[valid_mask])
            train_num += data[valid_mask].size(0)
            train_conf_num += data.size(0)
        lossOfepochInTrain.append(train_loss / train_num)
        accOfepochInTrain.append(train_correct / train_num)
        tqdm.write(
            f"epoch:{epoch + 1}/{epochs} train_loss:{train_loss / train_num:.4f} train_correct:{train_correct / train_num:.4f}  conf_correct:{train_conf_correct / train_conf_num:.4f}")
        if np.isnan(train_loss):
            nanLossCount += 1

        # 在验证集上验证
        with torch.no_grad():
            model.eval()
            hidden = None  #torch.zeros(hid0 * n_layers, bs, hs)
            for data, target, conf_target, cs, lattice, pg, _, abc, angles in tqdm(val_dataloader,
                                                                                   desc=f"{model_name} epoch:{epoch + 1}/{epochs}",
                                                                                   unit="batch", file=sys.stdout):
                data, target, conf_target, cs, lattice, pg, abc, angles = data.to(device), target.to(
                    device), conf_target.to(
                    device), cs.to(device), lattice.to(device), pg.to(device), abc.to(device), angles.to(device)
                # output = model(data)
                # output, hidden = model(data, hidden) #.detach()
                data = data.unsqueeze(1)
                # cs = torch.zeros_like(cs,dtype=torch.float32)
                # lattice = torch.zeros_like(lattice,dtype=torch.float32)
                # pg = torch.zeros_like(pg,dtype=torch.float32)
                output, conf, abc_angles = model(data, cs, lattice, pg)
                pre = torch.argmax(output, dim=1)
                conf_soma = torch.argmax(conf, dim=1)
                abc_angles_labels = torch.cat([abc, angles], dim=1)
                loss_abc_angles = criterion_abc_angles(abc_angles, abc_angles_labels) / (1000)
                loss = criterion(output, target)
                loss_conf = criterion_conf(conf, conf_target)
                loss_ = loss + loss_conf + loss_abc_angles
                val_loss += loss_.item() * data.size(0)
                valid_mask = (target >= start) & (target < start + length)
                val_conf_correct += torch.sum(conf_soma == conf_target)
                val_correct += torch.sum(pre[valid_mask] == target.data[valid_mask])
                val_num += data[valid_mask].size(0)
                val_conf_num += data.size(0)
        lossOfepochInVal.append(val_loss / val_num)
        accOfepochInVal.append(val_correct / val_num)
        epoch_infos = f"epoch:{epoch + 1}/{epochs} val_loss:{val_loss / val_num:.4f}  val_correct:{val_correct / val_num:.4f}  conf_correct:{val_conf_correct / val_conf_num}"
        # if np.isnan(val_loss):
        #     nanLossCount += 1
        # if nanLossCount >= 3:
        #     raise (Exception("There will not be a fourth time for nan loss."))
        # val_loss_diff = lossOfepochInVal[-1] - min_val_loss
        # if val_loss_diff > 2e2 * min_val_loss:
        #     print("Warning! Overfit happens... Now stop training.")
        #     break

        if ((val_correct / val_num) + (val_conf_correct / val_conf_num)) / 2 > best_avg_acc:
            tqdm.write(epoch_infos + " ... ModelSaving")
            best_avg_acc = ((val_correct / val_num) + (val_conf_correct / val_conf_num)) / 2
            early_stop = 30
            best_acc = accOfepochInVal[-1]
            min_val_loss = lossOfepochInVal[-1]
            best_model_wts = copy.deepcopy(model.module.state_dict()) if isinstance(model,
                                                                                    nn.DataParallel) else copy.deepcopy(
                model.state_dict())
            os.makedirs("subresnet 3attention 7 v7 angles,abc", exist_ok=True)
            torch.save(best_model_wts, f'subresnet 3attention 7 v7 angles,abc/{model_name} {news}.pth')
        else:
            tqdm.write(epoch_infos)
            early_stop -= 1

        if early_stop <= 0:
            print("Early stop!")
            break

    # 训练记录
    print(len(lossOfepochInTrain))
    train_process = pd.DataFrame(data={'epoch': range(1, epoch + 2),
                                       'loss in train': lossOfepochInTrain,
                                       'loss in val': lossOfepochInVal,
                                       })
    os.makedirs('Submodels_sg/train_process', exist_ok=True)
    train_process.to_csv(f'Submodels_sg/train_process_{model_name}-{news}.csv', index=False)

    return train_process


In [ ]:
n_in = 1500  # input length
n_out = 230  # target length # num_classes
input_size = 1  #  input dim # input size # num nodes # unit s
# batch_size = bs = 16 # batch_size
model_name = "model"
# train_(model_list[i], muti_name_list[i],pretrained=False)


In [ ]:
# model = Sub_BlockRNN_pg_muti(n_in, 32, bs, input_size, len(dvsid), hidden_dim=hs, n_rnn_layers=n_layers,
#                                      bidirectional=bidirect)
# model.load_state_dict(torch.load("/home/light/PycharmProjects/material/resampled_test/BlockRNN-mutipg_ba.pth"),
#                       strict=False)
def get_models(lengths):
    models = []
    for i, length in tqdm(enumerate(lengths)):
        model = RCNet(
            in_channels=1,
            base_filters=16,
            kernel_size=15,
            stride=2,
            groups=1,
            n_block=28,  #24,
            n_classes=230,
            downsample_gap=2,
            increasefilter_gap=4,
            use_bn=True,
            use_do=True,
            num_heads=2
        )
        if os.path.exists(f"subresnet 3attention 7 v7 angles,abc/model {i} angles_abc v7.pth"):
            model.load_state_dict(torch.load(f"subresnet 3attention 7 v7 angles,abc/model {i} angles_abc v7.pth"),
                                  strict=False)
            tqdm.write(f"Model{i} loading...")
        else:
            model.load_state_dict(torch.load(f"subresnet 3attention 7 v7 angles,abc/model {i} angles_abc v6.pth"),
                                  strict=False)
            # model.load_state_dict(
            #     torch.load(f"resnet-muti_sg250529.pth"), strict=False)
            tqdm.write(f"Model{i} creating...")
        models.append(model)
    return models


In [ ]:
starts = [0, 2, 15, 74, 142, 167, 194]
lengths = [starts[i] - starts[i - 1] for i in range(1, len(starts))]
lengths += [230 - 194]
print(lengths)

In [ ]:
train_root = "3attention_angle_sidelength/train_data_3attention.npy"
val_root = "3attention_angle_sidelength/val_data_3attention.npy"
test_root = "3attention_angle_sidelength/test_data_3attention.npy"
train_Datasets = []
val_Datasets = []
test_Datasets = []
num_lists = []

for i, starts_lengths in enumerate(zip(starts, lengths)):
    start, length = starts_lengths
    train_data = Sub_XRDDataset_3attention_angle_abc(train_root, cls='space_group', start=start, length=length)
    val_data = Sub_XRDDataset_3attention_angle_abc(val_root, cls='space_group', start=start, length=length)
    test_data = Sub_XRDDataset_3attention_angle_abc(test_root, cls='space_group', start=start, length=length)
    train_Datasets.append(train_data)
    val_Datasets.append(val_data)
    test_Datasets.append(test_data)
    num_list = train_data.get_num_list()
    num_list[np.isnan(num_list)] = max(num_list)
    num_lists.append(num_list)

# #


In [ ]:
models = get_models(lengths)

In [ ]:
# for i in range(0, len(lengths)):
for i in range(len(lengths)):
    model = models[i]
    train_data = train_Datasets[i]
    val_data = val_Datasets[i]
    train_loader = Data.DataLoader(dataset=train_data, batch_size=bs, shuffle=True, num_workers=16, drop_last=True,
                                   collate_fn=collate_fn)
    val_loader = Data.DataLoader(dataset=val_data, batch_size=bs, shuffle=True, num_workers=16, drop_last=True,
                                 collate_fn=collate_fn)
    train(model, train_loader, val_loader, "model " + str(i), num_lists[i], starts[i], lengths[i], 200)


In [ ]:
def test_conf_target(model, val_dataloader, start, length):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()  #FocalLoss(gamma=2)
    dvsid = [0, 1, 2, 3]
    model = nn.DataParallel(model, output_device=dvsid).to(device)  # model.to(device) #
    sg_raw = []
    sg_index_raw = []
    sg_conf = []
    abc_angles_list = []
    target_raw = []
    # best_acc = 0.0
    val_loss = 0
    global val_correct
    val_correct = 0
    val_num = 0
    conf_correct = 0
    conf_num = 0
    # 在验证集上验证
    with torch.no_grad():
        model.eval()
        for data, target, conf_label, cs, lattice, pg, xrd, abc, angles in tqdm(val_dataloader):
            data, target, conf_label, cs, lattice, pg, abc, angles = data.to(device), target.to(device), conf_label.to(
                device), cs.to(device), lattice.to(device), pg.to(device), abc.to(device), angles.to(
                device)  #data.to(device), target.to(device)
            data = data.unsqueeze(1)
            output, conf, abc_angles = model(data, cs, lattice, pg)
            output_ = torch.softmax(output, dim=1)
            o = torch.argmax(output, dim=1)
            conf_ = torch.argmax(conf, dim=1)
            valid_mask = (target >= start) & (target < start + length)
            val_correct += torch.sum(o[valid_mask] == target.data[valid_mask])

            conf_num += data[valid_mask].size(0)
            val_num += data[valid_mask].size(0)
            output_conf = conf
            output_max, output_index = torch.max(output_, dim=1)
            for sg_max, sg_index, c, abc_angles_sample, tar in zip(output_max, output_index, output_conf[:, 1],
                                                                   abc_angles, target):
                sg_raw.append(sg_max.item())
                sg_index_raw.append(sg_index.item())
                sg_conf.append(c.item())
                abc_angles_list.append(abc_angles_sample)
                target_raw.append(tar.item())
        tqdm.write(f"acc:{val_correct / val_num:.4f} conf_acc:{conf_correct / conf_num}")
    return sg_raw, sg_index_raw, sg_conf, abc_angles_list, target_raw


In [ ]:
models = get_models(lengths)

In [ ]:

test_root = "3attention_angle_sidelength/test_data_3attention.npy"
sg_mat = []
sg_index_mat = []
conf_mats = []
abc_angles_mat = []
target_mat = []
test_data = Sub_XRDDataset_3attention4test_angle_abc(test_root, cls='space_group', start=0, length=230)
for i in range(len(models)):
    modeltest = models[i]
    test_loader = Data.DataLoader(dataset=test_data, batch_size=bs, shuffle=False, num_workers=16, drop_last=True,
                                  collate_fn=collate_fn)
    sg_raw, sg_index_raw, sg_conf, abc_angles_list, target_raw = test_conf_target(modeltest, test_loader, starts[i],
                                                                                  lengths[i])
    sg_mat.append(sg_raw)
    sg_index_mat.append(sg_index_raw)
    conf_mats.append(sg_conf)
    abc_angles_mat.append(abc_angles_list)
    target_mat.append(target_raw)

In [ ]:
sg_mat_ = np.array(sg_mat)
sg_index_mat_ = np.array(sg_index_mat)
conf_mats_ = np.array(conf_mats)

target_mat_ = np.array(target_mat)


In [ ]:


d = []
acc_list = []
gamma_list = []
correct_list = [0] * len(lengths)
num_list = [0] * len(starts)
correct_list_ = [0] * 230
num_list_ = [0] * 230
gamma_list_ = [0] * 230
sg_acc_list = [0] * 230
max_acc = 0
max_gamma = 0
pre_list = []
max_idx = None
for gamma in [j / 50 for j in range(51)]:
    num = 0
    mat = np.zeros((230, 230))
    correct_num = 0
    output_mat = sg_mat_ * gamma + conf_mats_ * (1 - gamma)
    max_indices = np.argmax(output_mat, axis=0)
    for i in range(len(max_indices)):
        pre_list.append(sg_index_mat_[max_indices[i]][i])
        mat[sg_index_mat_[max_indices[i]][i], target_mat_[max_indices[i]][i]] += 1
        if sg_index_mat_[max_indices[i]][i] == target_mat_[max_indices[i]][i]:
            correct_num += 1
            d.append(sg_index_mat_[max_indices[i]][i])
            for idx, start_length in enumerate(zip(starts, lengths)):
                start, length = start_length
                if sg_index_mat_[max_indices[i]][i] >= start and sg_index_mat_[max_indices[i]][i] < start + length:
                    correct_list[idx] += 1
        for idx, start_length in enumerate(zip(starts, lengths)):
            start, length = start_length
            if target_mat_[max_indices[i]][i] >= start and target_mat_[max_indices[i]][i] < start + length:
                num_list[idx] += 1

        num += 1
    gamma_list.append(gamma)
    acc_list.append(correct_num / num)

    if correct_num / num > max_acc:
        max_acc = correct_num / num
        cn = correct_num
        n = num
        max_gamma = gamma
        best_mat = mat
        correct_np = np.array(correct_list)
        num_np = np.array(num_list)
        max_idx = max_indices


